# Atelier Prompt Engineering

## Partie 1 – Anatomie d'un prompt 

Construire un prompt, à partir de ses différentes composantes, pour résoudre un problème comme « Je souhaite analyser les retours de clients d'une entreprise. » 

![prompt Partie 1](images/partie1_prompt_1.png)
![prompt Partie 1](images/partie1_prompt_2.png)
![Réponse Partie 1](images/partie1_reponse.png)

## Partie 2 - Comparer les techniques de prompting

Tester puis évaluer les résultats obtenus avec les techniques zero-shot, one-shot, few-shot et un prompt structuré sur une demande du genre : «  Classer le commentaire suivant :   "Le service est rapide mais l'application plante régulièrement."   Classes possibles :    positif, négatif, neutre. » 

![Prompt Partie 2 avec zero-shot](images/partie2_zero-shot.png)

Réponse zero-shot : "Négatif", avec une justification spontanée (le LLM explique pourquoi malgré l'aspect positif "rapide"). Intéressant même sans qu'on le demande, il a choisi de justifier, ce sera à comparer avec les autres techniques.

![prompt Partie 2 avec one-shot](images/partie2_one-shot.png)

Le format a changé comme prévu : le LLM a imité le style de l'exemple donné ("Classe : X", court, sans explication) ça montre bien l'effet du one-shot sur la forme de la réponse.

![Prompt Partie 2 avec few-shot](images/partie2_few-shot.png)

Avec plusieurs exemples bien répartis entre les 3 classes (positif, négatif, neutre), le LLM revient sur "négatif", cohérent avec le zero-shot, mais cette fois sans justification (il a gardé le format court de l'exemple). Ça confirme l'hypothèse : le one-shot avait biaisé la réponse vers "neutre" à cause d'un exemple non représentatif, alors que le few-shot, avec des exemples plus variés, permet un jugement plus fiable* proche de l'intuition naturelle du zero-shot, mais avec un format plus contrôlé.

![Prompt Partie 2 avec structured prompting](images/partie2_structured-prompting.png)

| Technique | Classe | Justification | Format |
|---|---|---|---|
| Zero-shot | Négatif | Oui (spontanée) | Libre, développé |
| One-shot | Neutre | Non | Court, imité de l'exemple |
| Few-shot | Négatif | Non | Court, imité de l'exemple |
| Structuré | Négatif | Oui (demandée) | Précis, imposé |

## Évaluation
### Stabilité de la classification :
3 techniques sur 4 (zero-shot, few-shot, structuré) convergent vers "négatif". Seul le one-shot a dévié vers "neutre" probablement biaisé par l'unique exemple fourni ("cher mais bien fait" → neutre), qui n'était pas assez représentatif du cas à traiter. Leçon : un seul exemple mal choisi peut fausser le jugement du LLM davantage qu'aucun exemple du tout.
### Contrôle du format :
le zero-shot laisse le LLM libre (il choisit de justifier). Le one-shot et le few-shot imposent implicitement un format via l'exemple (réponse courte). Le prompt structuré est le seul à garantir explicitement le format ET la présence d'une justification, sans dépendre d'un exemple.
### Fiabilité générale :
le prompt structuré est la technique la plus fiable ici — il combine la justesse du zero-shot/few-shot (bonne classe) avec un format prévisible et une justification garantie, contrairement au one-shot qui s'est révélé le moins fiable des quatre sur ce cas précis.
### Cas ambigu révélateur :
ce commentaire mixte (aspect positif + négatif) était un bon choix de test il a permis de révéler que le nombre et la qualité des exemples influencent directement le jugement du LLM, pas seulement la forme de sa réponse.

## Partie 3 – Prompt Engineering et raisonnement

### 1) Décomposer le prompt suivant : 
« Analyse ces avis clients et donne-moi les problèmes les plus importants ainsi que les recommandations. » 

### Prompt

Analyse les avis clients suivants en suivant ces étapes :

1. Lis attentivement chaque avis client fourni.
2. Identifie les problèmes mentionnés dans chaque avis.
3. Regroupe les problèmes similaires et classe-les par ordre d'importance (fréquence ou gravité).
4. Pour chaque problème important identifié, propose une recommandation concrète.

Présente ta réponse en suivant ces 4 étapes, dans l'ordre, en indiquant clairement le résultat de chaque étape.

Avis clients à analyser :
{"input": "Livraison très en retard, aucune information pendant 5 jours, très déçu.", "output": 2}
{"input": "L'application plante souvent quand je veux payer, c'est frustrant.", "output": 3}
{"input": "Service client réactif et sympathique, ils ont résolu mon problème en 10 minutes.", "output": 9}
{"input": "Produit conforme à la description, rien à redire.", "output": 8}
{"input": "J'ai attendu 4 jours une réponse du support pour ma réclamation.", "output": 3}
{"input": "Très satisfait de la qualité, je recommande sans hésiter.", "output": 10}
{"input": "Le site web est lent et l'interface pas très claire.", "output": 4}
{"input": "Bon rapport qualité-prix, livraison rapide cette fois-ci.", "output": 8}
{"input": "Encore un bug sur l'application, ça devient récurrent.", "output": 2}
{"input": "Commande arrivée à temps, emballage soigné, rien à signaler.", "output": 9}

### Reponse

### Étape 1 — Lecture des avis et problèmes mentionnés

| Avis | Problème identifié |
|---|---|
| Livraison très en retard, aucune information pendant 5 jours | Retard de livraison + manque d’information |
| L’application plante lors du paiement | Bug de l’application lors du paiement |
| Service client réactif | Aucun problème signalé |
| Produit conforme | Aucun problème signalé |
| Réponse du support après 4 jours | Délai de réponse du support |
| Client très satisfait | Aucun problème signalé |
| Site lent et interface peu claire | Lenteur du site + interface peu claire |
| Livraison rapide | Aucun problème signalé |
| Bug récurrent de l’application | Bugs récurrents de l’application |
| Commande à temps et emballage soigné | Aucun problème signalé |

---

### Étape 2 — Regroupement des problèmes similaires

1. **Problèmes liés à l’application** — 2 avis
   - Plantage lors du paiement.
   - Bugs récurrents.

2. **Problèmes liés à la livraison et à la communication** — 1 avis
   - Livraison très en retard.
   - Absence d’information pendant 5 jours.

3. **Problèmes liés au support client** — 1 avis
   - Réponse après 4 jours.

4. **Problèmes liés au site web** — 1 avis
   - Site lent.
   - Interface peu claire.

---

### Étape 3 — Classement par importance

1. **Instabilité de l’application** — 2 avis, dont un problème affectant directement le paiement.
2. **Retard et manque de suivi des livraisons** — problème important en termes d’expérience client.
3. **Délais de réponse du support** — 4 jours pour une réclamation.
4. **Lenteur et manque de clarté du site** — problème d’utilisation mentionné dans un avis.

---

### Étape 4 — Recommandations concrètes

1. **Application**
   - Prioriser l’analyse et la correction des bugs, notamment ceux qui provoquent des plantages pendant le paiement.
   - Mettre en place des tests réguliers pour éviter la réapparition des bugs.

2. **Livraison**
   - Améliorer le suivi des commandes et informer automatiquement le client en cas de retard.
   - Éviter les périodes prolongées sans information sur l’état de la livraison.

3. **Support client**
   - Réduire le délai de traitement des réclamations et assurer un suivi plus rapide des demandes.

4. **Site web**
   - Optimiser les performances du site et clarifier l’interface afin de faciliter son utilisation.

### 2) Proposer au modèle un texte à analyser avec des contraintes puis utiliser un second prompt lui demandant de vérifier sa réponse précédente et de contrôler les informations non justifiées, les contradictions, les informations absentes, les éventuelles hallucinations et le respect des contraintes. 

## Prompt

Vérifie la réponse précédente à l'analyse des avis clients en répondant aux points suivants :

1. Y a-t-il des informations non justifiées par les avis clients fournis ?
2. Y a-t-il des contradictions dans l'analyse ?
3. Y a-t-il des informations importantes absentes ou oubliées ?
4. Y a-t-il des hallucinations (informations inventées, absentes des avis d'origine) ?
5. Les contraintes initiales ont-elles été respectées (rester factuel, ne pas inventer, structurer par étapes) ?

Réponds point par point pour chacun de ces 5 axes.

## Reponse

## 1. Informations non justifiées par les avis clients

**Oui, légèrement.**

- La phrase **« problème important en termes d’expérience client »** est une interprétation générale. Elle est plausible, mais elle n'est pas directement exprimée dans les avis.
- La recommandation de **« prioriser »** certains problèmes constitue une proposition d'analyse et non un fait directement fourni par les clients.
- Le reste des informations est globalement directement justifié par les commentaires.

## 2. Contradictions dans l’analyse

**Non, il n’y a pas de contradiction majeure.**

- Les deux avis concernant l’application signalent bien des problèmes : plantage au paiement et bugs récurrents.
- Les avis positifs sur la livraison (« livraison rapide », « commande arrivée à temps ») ne contredisent pas le retard signalé dans un autre avis. Ils montrent simplement des expériences différentes.
- Le support client présente toutefois des expériences contrastées : un client indique une résolution en 10 minutes, tandis qu’un autre a attendu 4 jours pour une réponse.

## 3. Informations importantes absentes ou oubliées

**Oui, quelques éléments peuvent être améliorés.**

- Il aurait été pertinent de signaler explicitement que **les expériences concernant le support client sont contrastées**.
- Les avis positifs concernant la **qualité et la conformité des produits** sont peu exploités dans l'analyse.
- Il aurait également été utile de préciser que **les problèmes liés à l’application sont les seuls problèmes mentionnés dans plusieurs avis**, avec 2 avis concernés.

## 4. Hallucinations

**Il n’y a pas d’hallucination importante.**

Les principaux éléments correspondent bien aux avis fournis :

- **5 jours sans information** → présent dans l’avis sur la livraison.
- **4 jours d’attente du support** → présent dans l’avis concernant la réclamation.
- **2 problèmes liés à l’application** → présents dans deux avis différents.
- **Site lent et interface peu claire** → présents dans l’avis correspondant.
- **Livraison rapide ou à temps** → présentes dans les avis positifs.

Les recommandations comme **« mettre en place des tests réguliers »** ou **« informer automatiquement le client »** sont des propositions et non des informations présentées comme des faits. Elles ne constituent donc pas des hallucinations.

## 5. Respect des contraintes initiales

**Oui, globalement les contraintes ont été respectées.**

- **Rester factuel :** globalement respecté.
- **Ne pas inventer :** respecté ; aucune donnée externe ou statistique inventée n’a été ajoutée.
- **Identifier les problèmes :** respecté.
- **Regrouper les problèmes similaires :** respecté.
- **Classer les problèmes par importance :** respecté, même si le classement pourrait être davantage justifié par la fréquence et la gravité.
- **Proposer des recommandations concrètes :** respecté.
- **Structurer la réponse en 4 étapes :** respecté.

## Conclusion

La réponse est **globalement correcte et fidèle aux données fournies**.

Les principales améliorations seraient de :

1. mieux faire ressortir les **avis positifs** ;
2. signaler explicitement la **divergence concernant le support client** ;
3. distinguer plus clairement les **faits observés** des **interprétations et recommandations**.

## Partie 4 – Sorties structurées

### Le LLM retourne : « Le commentaire semble plutôt négatif. Le client est mécontent du délai de livraison... » 
#### 1) Pour rendre cette réponse plus facile à exploiter par une application, créer un prompt demandant du JSON avec les champs sentiment, categorie, urgence, probleme, confiance. 
Préciser les types des valeurs et les valeurs autorisées
Exemple de résultat 
```json
{
    "sentiment": "negatif",
    "categorie": "livraison",
    "urgence": "moyenne", 
    "probleme": "Retard de livraison", 
    "confiance": 0.91 
}
```

### Prompt

Analyse le commentaire client suivant et retourne le résultat **uniquement** au format JSON, avec les champs suivants :

- `sentiment` (string) : "positif", "negatif" ou "neutre"
- `categorie` (string) : le thème principal du commentaire (ex. "livraison", "application", "support", "produit")
- `urgence` (string) : "faible", "moyenne" ou "élevée"
- `probleme` (string) : description courte du problème principal identifié
- `confiance` (number) : score entre 0 et 1 représentant ton degré de certitude dans cette analyse

Ne retourne rien d'autre que le JSON.

Commentaire à analyser : "Le commentaire semble plutôt négatif. Le client est mécontent du délai de livraison..."

### Reponse

```json
{
  "sentiment": "negatif",
  "categorie": "livraison",
  "urgence": "moyenne",
  "probleme": "Délai de livraison jugé insatisfaisant.",
  "confiance": 0.98
}
```

#### 2) Ajouter une validation en définissant les règles de sortie suivantes : format JSON valide ; aucune propriété supplémentaire ; sentiment vaut positif, negatif ou neutre ; confiance est compris entre 0 et 1 ; urgence vaut faible, moyenne ou élevée. 

### Prompt

Analyse le commentaire client suivant et retourne le résultat *uniquement* au format JSON, en respectant strictement les règles suivantes :

Le JSON doit être valide (syntaxe correcte, pas de texte autour).
Aucune propriété supplémentaire ne doit être ajoutée en dehors des 5 champs suivants.
- `sentiment` (string) : doit valoir exactement "positif", "negatif" ou "neutre".
- `categorie` (string) : thème principal du commentaire.
- `urgence` (string) : doit valoir exactement "faible", "moyenne" ou "élevée".
- `probleme` (string) : description courte du problème principal.
- `confiance` (number) : doit être compris entre 0 et 1.

Commentaire à analyser : "Le commentaire semble plutôt négatif. Le client est mécontent du délai de livraison..."

### Reponse

{"sentiment":"negatif","categorie":"livraison","urgence":"moyenne","probleme":"Délai de livraison insatisfaisant.","confiance":0.99}

#### Comparaison des deux versions

| Aspect | Version 1 (sans règles) | Version 2 (avec règles) |
|---|---|---|
| Format | JSON indenté (lisible) | JSON sur une ligne (compact) |
| Champs | 5, corrects | 5, corrects |
| Confiance	| 0.98	| 0.99 |
| Propriétés en trop | Aucune | Aucune |

Résultat : JSON valide, format compact, tous les champs corrects, confiance 0.99.

À retenir :

Le format a changé (indenté → compact) sans qu'on l'ait demandé, si on veut un format visuel stable, il faut le préciser explicitement.
Le score de confiance reste très élevé (0.98→0.99) sans méthode de calcul claire, à noter comme limite (indicatif, pas vraiment fiable).
Les règles de validation (pas de champ en trop, valeurs autorisées) sont respectées dans les deux versions.

## Partie 5 – Prompts pour les applications métier

### 1) Créer un prompt permettant de résumer un document avec les contraintes : maximum 250 mots ; conserver les informations factuelles ; identifier les objectifs ; identifier les résultats ; identifier les recommandations ; n'inventer aucune information. 

### Prompt

Résume le texte suivant en respectant les contraintes suivantes :

- Maximum 250 mots.
- Conserve uniquement les informations factuelles présentes dans le texte.
- Identifie clairement les objectifs mentionnés.
- Identifie clairement les résultats ou constats mentionnés.
- Identifie clairement les recommandations ou bonnes pratiques mentionnées.
- N'invente aucune information absente du texte.

Texte à résumer :

Le prompt engineering est une discipline apparue avec la démocratisation des grands modèles de langage (LLM). Elle consiste à concevoir des instructions, appelées prompts, permettant d'orienter le comportement d'un modèle d'intelligence artificielle afin d'obtenir des réponses pertinentes, précises et exploitables.

Contrairement à une idée reçue, le prompt engineering ne se limite pas à poser une question de façon claire. Il s'agit d'une approche structurée qui prend en compte plusieurs éléments : le rôle attribué au modèle, le contexte de la demande, la tâche précise à accomplir, les contraintes à respecter, ainsi que le format de sortie attendu. Cette structuration permet de réduire l'ambiguïté et d'améliorer la fiabilité des réponses produites.

Plusieurs techniques existent pour affiner un prompt. Le zero-shot consiste à ne fournir aucun exemple, laissant le modèle interpréter seul la demande. Le few-shot, à l'inverse, propose plusieurs exemples afin de guider le raisonnement du modèle vers un format ou un résultat attendu. D'autres approches, comme le raisonnement par étapes (chain-of-thought), encouragent le modèle à décomposer un problème complexe en sous-étapes avant de fournir une conclusion.

Le prompt engineering trouve aujourd'hui de nombreuses applications concrètes en entreprise : analyse de retours clients, génération de résumés, classification automatique de documents, extraction d'informations structurées, ou encore assistance à la rédaction. À mesure que les modèles de langage évoluent, la maîtrise du prompt engineering devient une compétence clé pour exploiter efficacement ces outils dans un cadre professionnel.

### Reponse

#### Objectif
Le prompt engineering consiste à concevoir des instructions structurées pour orienter le comportement des modèles d’intelligence artificielle et obtenir des réponses pertinentes, précises et exploitables.

#### Constats
- Le prompt engineering ne se limite pas à poser une question clairement.
- Il prend en compte le **rôle**, le **contexte**, la **tâche**, les **contraintes** et le **format de sortie**.
- Cette structuration réduit l’ambiguïté et améliore la fiabilité des réponses.
- Le **zero-shot** utilise aucun exemple, tandis que le **few-shot** fournit plusieurs exemples pour guider le modèle.
- Le **raisonnement par étapes (chain-of-thought)** encourage la décomposition des problèmes complexes en sous-étapes.
- Le prompt engineering est utilisé notamment pour l’analyse de retours clients, les résumés, la classification de documents, l’extraction d’informations structurées et l’assistance à la rédaction.

#### Recommandations / bonnes pratiques
- Structurer les prompts en précisant le rôle, le contexte, la tâche, les contraintes et le format attendu.
- Utiliser des exemples lorsque cela permet de guider le modèle vers un résultat ou un format attendu.
- Pour les problèmes complexes, décomposer la tâche en plusieurs sous-étapes.

#### Conclusion
Avec l’évolution des modèles de langage, la maîtrise du prompt engineering devient une compétence clé pour exploiter efficacement ces outils dans un cadre professionnel.

### Analyse
Le résumé produit respecte les contraintes posées : il reste factuel, ne contient aucune information absente du texte source, et sa longueur est largement inférieure à la limite de 250 mots.

Un point notable dans le comportement du LLM : le texte source n'étant pas structuré comme une étude (il ne contient pas d'"objectifs" ou de "résultats" au sens strict, mais plutôt des définitions et explications), le LLM n'a pas forcé artificiellement du contenu dans chaque catégorie demandée. Il a réinterprété "résultats" en "constats", ce qui reste fidèle à la nature du texte plutôt que de générer une information inventée pour combler la catégorie.

Cela dit, ce comportement soulève une limite : le LLM a silencieusement adapté les catégories sans le signaler explicitement. Une personne qui n'aurait pas lu le texte source pourrait ne pas se rendre compte que la catégorie "résultats" n'existait pas réellement dans le texte d'origine.

Le test sur la contrainte de longueur (250 mots) reste également peu concluant, dans la mesure où le texte source lui-même ne faisait qu'environ 235 mots la capacité du LLM à condenser un texte long n'a donc pas été réellement sollicitée ici.

### 2) Créer un prompt permettant de traduire un document français en anglais avec les contraintes : conserver le sens ; conserver la structure ; conserver les termes techniques ; ne pas résumer ; ne rajoute aucune information 

### Prompt

Traduis le texte suivant du français vers l'anglais, en respectant strictement les contraintes suivantes :

- Conserve fidèlement le sens du texte original.
- Conserve la structure (mêmes paragraphes, même organisation des idées).
- Conserve les termes techniques tels quels ou avec leur équivalent anglais standard (ex. "prompt engineering", "zero-shot", "few-shot", "chain-of-thought").
- Ne résume pas le texte, traduis-le intégralement.
- N'ajoute aucune information absente du texte original.

Texte à traduire :

Le prompt engineering est une discipline apparue avec la démocratisation des grands modèles de langage (LLM). Elle consiste à concevoir des instructions, appelées prompts, permettant d'orienter le comportement d'un modèle d'intelligence artificielle afin d'obtenir des réponses pertinentes, précises et exploitables.

Contrairement à une idée reçue, le prompt engineering ne se limite pas à poser une question de façon claire. Il s'agit d'une approche structurée qui prend en compte plusieurs éléments : le rôle attribué au modèle, le contexte de la demande, la tâche précise à accomplir, les contraintes à respecter, ainsi que le format de sortie attendu. Cette structuration permet de réduire l'ambiguïté et d'améliorer la fiabilité des réponses produites.

Plusieurs techniques existent pour affiner un prompt. Le zero-shot consiste à ne fournir aucun exemple, laissant le modèle interpréter seul la demande. Le few-shot, à l'inverse, propose plusieurs exemples afin de guider le raisonnement du modèle vers un format ou un résultat attendu. D'autres approches, comme le raisonnement par étapes (chain-of-thought), encouragent le modèle à décomposer un problème complexe en sous-étapes avant de fournir une conclusion.

Le prompt engineering trouve aujourd'hui de nombreuses applications concrètes en entreprise : analyse de retours clients, génération de résumés, classification automatique de documents, extraction d'informations structurées, ou encore assistance à la rédaction. À mesure que les modèles de langage évoluent, la maîtrise du prompt engineering devient une compétence clé pour exploiter efficacement ces outils dans un cadre professionnel.

### Reponse

Prompt engineering is a discipline that emerged with the democratization of large language models (LLMs). It consists of designing instructions, called prompts, that make it possible to guide the behavior of an artificial intelligence model in order to obtain relevant, accurate, and usable responses.

Contrary to a common misconception, prompt engineering is not limited to asking a question clearly. It is a structured approach that takes into account several elements: the role assigned to the model, the context of the request, the specific task to be performed, the constraints to be respected, as well as the expected output format. This structuring helps reduce ambiguity and improve the reliability of the responses produced.

Several techniques exist to refine a prompt. Zero-shot consists of providing no examples, leaving the model to interpret the request on its own. Few-shot, on the other hand, provides several examples in order to guide the model's reasoning toward an expected format or result. Other approaches, such as step-by-step reasoning (chain-of-thought), encourage the model to break down a complex problem into sub-steps before providing a conclusion.

Prompt engineering currently has many practical applications in business: analysis of customer feedback, generation of summaries, automatic document classification, extraction of structured information, and assistance with writing. As language models evolve, mastering prompt engineering is becoming a key skill for effectively using these tools in a professional setting.

### Analyse

La traduction respecte les contraintes demandées : le sens du texte est fidèlement rendu, la structure (mêmes paragraphes, même enchaînement des idées) est conservée, et rien n'a été résumé ni ajouté par rapport au texte source.

Les termes techniques ("zero-shot", "few-shot", "chain-of-thought", "prompt engineering") sont bien conservés en anglais, ce qui est cohérent puisque ce sont déjà des termes standards du domaine. Le LLM a aussi bien distingué le concept traduisible ("step-by-step reasoning") du terme technique figé qu'il garde entre parenthèses ("chain-of-thought").

### 3) Créer un prompt permettant de classer un ticket informatique dans une catégorie (réseau, logiciel, matériel, sécurité, accès, autre). La réponse est au format JSON avec les champs categorie et justification 

### Prompt

Classe le ticket informatique suivant dans l'une des catégories suivantes : réseau, logiciel, matériel, sécurité, accès, autre.

Retourne ta réponse **uniquement** au format JSON avec les champs suivants :

- `categorie` (string) : une des 6 catégories listées ci-dessus.
- `justification` (string) : une phrase expliquant pourquoi ce ticket appartient à cette catégorie.

Ticket : "Depuis ce matin, je n'arrive plus à me connecter à mon compte professionnel, le système me dit que mon mot de passe est incorrect alors que je n'en ai pas changé. J'ai besoin d'accéder à mes fichiers rapidement pour une réunion."

### Reponse

``` Json
{
  "categorie": "accès",
  "justification": "Le ticket concerne l'impossibilité de se connecter au compte professionnel en raison d'un problème de mot de passe."
}
```

### Analyse

Le résultat est correct sur le fond : catégorie "accès" bien choisie (cohérent, puisque le problème central est l'impossibilité de se connecter, plutôt qu'une intrusion ou une faille de sécurité), avec une justification claire et directement liée au texte du ticket.

Concernant la forme : c'est le même comportement que dans la Partie 4 — le LLM retourne un JSON compact (tout sur une ligne) plutôt qu'indenté, alors qu'on n'a pas précisé de règle sur la mise en forme visuelle. Pour rappel, ça reste du JSON parfaitement valide (une application peut le lire sans problème), simplement moins lisible à l'œil pour un humain. Si tu veux un format indenté systématique, il faut l'ajouter explicitement dans le prompt (ex. "formate le JSON avec une indentation de 2 espaces").

### 4) Créer un prompt permettant d’extraire, à partir d’une facture, les infos (numéro_facture, date, client, montant_ht, tva, montant_ttc). Retourner uniquement un JSON valide avec null si une info est absente 

### Prompt

Extrais les informations suivantes de la facture ci-dessous, et retourne le résultat **uniquement** au format JSON avec les champs suivants :

- `numero_facture` (string)
- `date` (string)
- `client` (string)
- `montant_ht` (number)
- `tva` (string ou number, selon comme elle est indiquée)
- `montant_ttc` (number)

Si une information est absente de la facture, indique `null` pour ce champ. Ne calcule ni n'invente aucune valeur manquante.

Facture :

"Facture n° FA-2026-0347. Date : 15/08/2026. Client : Société Diallo & Fils. Montant HT : 1 200,00 €. TVA : 18%. Montant TTC non indiqué."

### Reponse

```Json
    {
        "numero_facture": "FA-2026-0347",
        "date": "15/08/2026",
        "client": "Société Diallo & Fils",
        "montant_ht": 1200,
        "tva": "18%",
        "montant_ttc": null
    }
```

### Analyse

Résultat correct sur le fond : toutes les infos présentes sont bien extraites, et surtout, le point clé du test est respecté le montant_ttc est bien mis à null plutôt que d'être calculé automatiquement (1200 × 1.18). Le LLM a donc bien suivi la consigne "ne pas inventer/calculer une valeur manquante".

Sur la forme : effectivement, cette fois le JSON est indenté alors qu'en Partie 4 et 5.3 il était compact sans qu'on ait rien précisé de différent dans le prompt. Ça confirme une observation à noter : le format visuel du JSON (indenté ou non) n'est pas stable d'une requête à l'autre, même avec des prompts similaires. Si tu veux un format garanti, il faut vraiment le spécifier explicitement à chaque fois (ex. "indente le JSON avec 2 espaces").

### 5) Créer un prompt permettant de rédiger un email destiné à un client dont la livraison a pris du retard avec les objectifs (reconnaître le retard, présenter des excuses, expliquer la situation sans inventer de cause et proposer une solution), le ton (professionnel, courtois et rassurant) et la longueur maximale (150 mots) 

### Prompt

Rédige un email destiné à un client dont la livraison a pris du retard, en respectant les éléments suivants :

**Objectifs** :

- Reconnaître le retard.
- Présenter des excuses.
- Expliquer la situation, sans inventer de cause précise si elle n'est pas connue.
- Proposer une solution ou un geste commercial.

**Ton** : professionnel, courtois et rassurant.

**Contrainte** : maximum 150 mots.

### Reponse

**Objet : Retard de livraison de votre commande**

Bonjour,

Nous vous informons que votre commande connaît actuellement un retard de livraison. Nous sommes sincèrement désolés pour ce contretemps et pour la gêne occasionnée.

Nous comprenons que ce retard puisse être décevant et vous remercions pour votre patience. À ce stade, nous ne disposons pas d'informations suffisantes pour vous communiquer une cause précise.

Nous suivons attentivement la situation afin de permettre la livraison de votre commande dans les meilleurs délais. À titre commercial, nous souhaitons vous proposer [un geste commercial / une remise / les frais de livraison offerts] afin de nous excuser pour ce désagrément.

Nous restons à votre disposition pour toute question complémentaire.

Cordialement,

L'équipe Service Client

### Analyse

Les 4 objectifs sont bien respectés : le retard est reconnu, des excuses sont présentées, la situation est expliquée sans inventer de cause précise, et une solution commerciale est proposée. Le ton est bien professionnel, courtois et rassurant.

Point intéressant à noter : plutôt que d'inventer un geste commercial précis, le LLM a laissé un espace à compléter ("[un geste commercial / une remise / les frais de livraison offerts]") — il a respecté la consigne "sans inventer" de façon très stricte, en évitant même de choisir arbitrairement une compensation, alors que ça n'était pas explicitement demandé pour cette partie-là. C'est une bonne illustration de prudence du modèle, mais ça montre aussi que l'email n'est pas directement utilisable tel quel sans complément humain.

La longueur reste bien sous la limite de 150 mots.

## Partie 6 – Prompt Engineering pour le Machine Learning 

### 1) Créer un prompt permettant, à partir du dataset des capteurs, de proposer des stratégies (détection, traitement, risques associés) de traitement des valeurs manquantes, des doublons, des valeurs aberrantes, des variables catégorielles 

#### connexion API : 
Cette cellule charge la clé Gemini stockée dans le fichier .env (sans l'écrire en clair dans le code), configure la librairie genai avec cette clé pour authentifier toutes les requêtes futures, puis crée un objet model représentant le modèle Gemini choisi (gemini-1.5-flash) — cet objet sera réutilisé à chaque envoi de prompt.

In [12]:
from dotenv import load_dotenv
import os
import google.generativeai as genai

load_dotenv()
genai.configure(api_key=os.getenv("GEMINI_API_KEY"))
model = genai.GenerativeModel("gemini-3.5-flash")

#### lecture des données : 
Cette cellule charge le fichier CSV dans un tableau pandas (df), puis en extrait un échantillon aléatoire de 15 lignes (echantillon) pour ne pas envoyer les 600+ lignes complètes dans un prompt. Le paramètre random_state=42 garantit que le même échantillon est tiré à chaque exécution, pour rester reproductible. La dernière ligne (echantillon seul) affiche ce tableau de façon lisible dans le notebook.

In [13]:
import pandas as pd

df = pd.read_csv("data/mesures_capteurs.csv")
echantillon = df.sample(15, random_state=42)
echantillon

,id_mesure,date_heure,id_capteur,batiment,temperature,humidite,pression,consommation,etat
412,M0315,2026-01-18 02:00:00,C003,B001,26.32,64.61,1026.92,158.30,OK
289,M0027,2026-01-06 02:00:00,C003,B001,21.20,62.95,1005.40,164.74,OK
76,M0189,2026-01-12 20:00:00,C009,B003,19.95,85.49,1027.28,201.12,ALERTE
78,M0105,2026-01-09 08:00:00,C009,B003,26.27,78.28,1012.45,210.91,OK
182,M0564,2026-01-28 11:00:00,C012,B004,29.03,41.19,1021.27,226.79,OK
594,M0459,2026-01-24 02:00:00,C003,B001,20.65,69.15,1017.63,183.96,OK
10,M0071,2026-01-07 22:00:00,C011,B004,28.44,47.55,1020.42,300.45,OK
131,M0085,2026-01-08 12:00:00,C001,B001,21.99,68.12,1022.24,207.39,OK
443,M0134,2026-01-10 13:00:00,C002,B001,22.65,69.52,1000.47,208.69,OK
86,M0040,2026-01-06 15:00:00,C004,B002,26.39,74.83,1009.01,176.68,OK


Cette cellule construit le texte du prompt en y insérant l'échantillon de données converti en texte brut **(echantillon.to_string()**), envoie ce prompt au modèle Gemini via **model.generate_content()**, puis affiche uniquement le texte de la réponse obtenue (response.text).

In [14]:
prompt = f"""Voici un extrait d'un dataset de capteurs mesurant la consommation énergétique d'un bâtiment, avec les colonnes suivantes : id_mesure, date_heure, id_capteur, batiment, temperature, humidite, pression, consommation, etat.

Pour ce dataset, propose des stratégies pour traiter les éléments suivants. Pour chacun, indique : la méthode de détection, la méthode de traitement, et les risques associés si mal géré :
1. Valeurs manquantes
2. Doublons
3. Valeurs aberrantes
4. Variables catégorielles

Voici un échantillon du dataset :
{echantillon.to_string(index=False)}
"""

response = model.generate_content(prompt)
print(response.text)

Voici une proposition de stratégie globale et adaptée à ce type de données industrielles et temporelles (IoT / Smart Building) pour chacun des quatre points.

---

### 1. Valeurs manquantes

Dans un contexte de capteurs de bâtiment, les pannes de réseau, les coupures d'électricité ou les opérations de maintenance génèrent fréquemment des données manquantes.

*   **Méthode de détection :**
    *   **Analyse globale :** Utilisation de fonctions comme `df.isnull().sum()` (Python) pour identifier les colonnes impactées.
    *   **Visualisation temporelle :** Utiliser la librairie `missingno` (Heatmap) pour voir si les manques sont aléatoires (un capteur isolé) ou systématiques (tous les capteurs coupés en même temps = panne générale).
*   **Méthode de traitement :**
    *   *Pour les variables météorologiques (Température, Humidité, Pression) :* Utiliser l'**interpolation temporelle** (linéaire ou spline) car le climat varie de manière continue.
    *   *Pour la variable cible (`consommati

La réponse répond bien aux 4 éléments demandés (valeurs manquantes, doublons, valeurs aberrantes, variables catégorielles), avec pour chacun une méthode de détection, une méthode de traitement, et des risques associés — donc la structure demandée dans le prompt est respectée.

### 2) Créer un prompt permettant, à partir du dataset des capteurs, de proposer les visualisations les plus pertinentes pour comprendre la consommation énergétique d'un bâtiment. Et pour chaque visualisation indiquer le type de graphique, les variables utilisées, l’objectif et l’interprétation attendue. 

In [15]:
prompt = f"""Voici un extrait d'un dataset de capteurs mesurant la consommation énergétique d'un bâtiment, avec les colonnes suivantes : id_mesure, date_heure, id_capteur, batiment, temperature, humidite, pression, consommation, etat.

Propose les visualisations les plus pertinentes pour comprendre la consommation énergétique du bâtiment. Pour chaque visualisation, indique :
1. Le type de graphique
2. Les variables utilisées
3. L'objectif de la visualisation
4. L'interprétation attendue

Voici un échantillon du dataset :
{echantillon.to_string(index=False)}
"""

response = model.generate_content(prompt)
print(response.text)

Pour analyser et comprendre efficacement la consommation énergétique de vos bâtiments à partir de ce dataset, voici les 5 visualisations les plus pertinentes. Elles permettent de couvrir les aspects temporels, comparatifs, corrélatifs et opérationnels.

---

### 1. L'évolution temporelle de la consommation par bâtiment
*   **Type de graphique :** Graphique linéaire (Line Chart) multi-séries.
*   **Variables utilisées :**
    *   Axe X : `date_heure` (agrégé par heure ou par jour)
    *   Axe Y : `consommation` (somme ou moyenne)
    *   Couleur / Légende : `batiment`
*   **Objectif de la visualisation :** Suivre la tendance de la consommation dans le temps et comparer le comportement des différents bâtiments sur une même période.
*   **Interprétation attendue :** Identification des cycles journaliers (pics en journée, creux la nuit), des variations le week-end, et détection visuelle des bâtiments qui surconsomment globalement par rapport aux autres (ex: B004 semble avoir des valeurs él

Le prompt structuré donne une réponse conforme et exploitable : chaque visualisation proposée respecte les 4 éléments demandés (type de graphique, variables, objectif, interprétation attendue), sans en oublier un seul sur les exemples visibles.

Point notable : le LLM ne se limite pas à des graphiques génériques il exploite le contenu réel du dataset pour rendre l'interprétation concrète, par exemple en citant directement une valeur observée dans l'échantillon ("B004 semble avoir des valeurs élevées comme 307.97") plutôt qu'une explication abstraite. Il propose aussi une visualisation qui croise la variable catégorielle etat avec la consommation, ce qui montre qu'il a bien intégré toutes les colonnes du dataset dans ses propositions, pas seulement les plus évidentes (température/consommation).

### 3) Créer un prompt permettant, à partir du dataset des capteurs, de proposer plusieurs modèles adaptés à la prédiction de la consommation énergétique d'un bâtiment. Pour chaque modèle, le LLM doit indiquer le principe, les avantages, les limites, le type de problème et les métriques pertinentes. 

In [16]:
prompt = f"""Voici un extrait d'un dataset de capteurs mesurant la consommation énergétique d'un bâtiment, avec les colonnes suivantes : id_mesure, date_heure, id_capteur, batiment, temperature, humidite, pression, consommation, etat.

Propose plusieurs modèles adaptés à la prédiction de la consommation énergétique du bâtiment. Pour chaque modèle, indique :
1. Le principe
2. Les avantages
3. Les limites
4. Le type de problème (classification, régression...)
5. Les métriques pertinentes

Voici un échantillon du dataset :
{echantillon.to_string(index=False)}
"""

response = model.generate_content(prompt)
print(response.text)

Pour prédire la consommation énergétique d'un bâtiment (variable continue `consommation`), nous sommes face à un problème de **régression**. De plus, la présence de la colonne `date_heure` indique une dimension temporelle forte. 

Avant d'appliquer les modèles, un travail de **Feature Engineering** (préparation des données) est indispensable :
*   **Données temporelles (`date_heure`)** : Extraire l'heure de la journée (crucial pour la consommation), le jour de la semaine (semaine vs week-end), et le mois.
*   **Données catégorielles (`batiment`, `id_capteur`, `etat`)** : Les encoder (One-Hot Encoding ou Target Encoding).

Voici 4 modèles adaptés à ce problème, du plus simple au plus complexe.

---

### 1. Régression Linéaire Régularisée (Ridge / Lasso)

#### 1. Le principe
Ce modèle cherche à établir une relation linéaire entre les variables explicatives (température, humidité, heure encodée, etc.) et la variable cible (consommation). Les régularisations Ridge (L2) ou Lasso (L1) ajoute

Réponse solide et bien structurée : le LLM identifie correctement le type de problème (régression, avec dimension temporelle) avant même de proposer les modèles, et couvre bien les 5 éléments demandés pour chaque modèle (principe, avantages, limites, type de problème, métriques).

Point notable : il propose une progression de complexité (du plus simple régression linéaire au plus avancé LightGBM/XGBoost), et ajoute une étape de préparation des données (feature engineering sur les variables temporelles et catégorielles) non demandée explicitement, mais pertinente puisque directement liée au dataset fourni. Il conclut aussi par une recommandation pratique (commencer par une baseline, puis complexifier).